# Pipeline de vendas com Apache Spark e arquitetura Medallion

**Laboratório completo | Bronze → Silver → Gold | 1.000.000 vendas, 250.000 clientes e 100.000 produtos**

Objetivos: gerar dados sintéticos escaláveis, ingerir arquivos em uma lakehouse Delta, validar e tratar dados, construir dimensões e fatos, produzir indicadores analíticos e executar verificações de qualidade.

**Ambiente:** Python 3.10+, Java 17, PySpark 3.5.x e Delta Lake 3.2.x. Recomenda-se **8–16 GB de RAM**, dependendo da máquina. Execute as células na ordem. O processamento é local por padrão; em clusters, ajuste caminhos e configurações.

> Os dados são sintéticos e não representam vendas reais. A geração usa expressões distribuídas do Spark, sem criar um milhão de linhas na memória do Python.

## 0. Preparação do ambiente

Em um ambiente limpo, instale os pacotes abaixo **antes** de iniciar o kernel. Em Databricks ou ambientes Spark gerenciados, utilize o runtime com Delta já configurado e ignore a instalação.

In [ ]:
%pip install pyspark==3.5.3 delta-spark==3.2.1 matplotlib pandas

**Atenção:** se o Spark já estiver iniciado, reinicie o kernel após a instalação. Em notebook local, a primeira inicialização do Delta pode baixar dependências Maven (necessita acesso à internet).

In [ ]:
import os
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

ROOT = str(Path.cwd() / "lakehouse_vendas")
RAW = f"{ROOT}/landing"
BRONZE = f"{ROOT}/bronze"
SILVER = f"{ROOT}/silver"
GOLD = f"{ROOT}/gold"
for path in (RAW, BRONZE, SILVER, GOLD):
    os.makedirs(path, exist_ok=True)

builder = (SparkSession.builder.appName("LakehouseVendasMedallion")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.memory", "6g"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version, "| diretório:", ROOT)

## 1. Desenho da arquitetura

```text
Spark (gerador sintético)
   │
   ▼
Landing / Parquet (clientes, produtos, vendas)
   │
   ▼
BRONZE / Delta: cópia tipada, rastreabilidade, timestamp de ingestão
   │
   ▼
SILVER / Delta: normalização, validação, deduplicação, integridade referencial
   │
   ▼
GOLD / Delta: fato de vendas + dimensões + agregações de negócio
   │
   ▼
Consultas Spark SQL, KPIs e visualizações
```

**Grão:** uma linha de venda corresponde a um item vendido (uma transação por `venda_id`). `valor_total = quantidade × preco_unitario × (1 − desconto_pct)`; receita considera o valor líquido de desconto, sem impostos ou devoluções.

A Bronze preserva os registros de entrada e metadados de ingestão. A Silver aplica regras e quarentena. A Gold disponibiliza modelos para consumo analítico.

## 2. Geração distribuída de dados sintéticos

In [ ]:
N_VENDAS = 1_000_000
N_CLIENTES = 250_000
N_PRODUTOS = 100_000
SEED = 42

clientes = (spark.range(1, N_CLIENTES + 1).withColumnRenamed("id", "cliente_id")
    .withColumn("nome", F.concat(F.lit("Cliente "), F.col("cliente_id")))
    .withColumn("email", F.concat(F.lit("cliente"), F.col("cliente_id"), F.lit("@exemplo.com")))
    .withColumn("estado", F.element_at(F.array(*[F.lit(x) for x in ["SP","RJ","MG","RS","PR","BA","SC","PE"]]),
                                    (F.pmod(F.col("cliente_id"), F.lit(8)) + 1).cast("int")))
    .withColumn("data_cadastro", F.date_add(F.lit("2022-01-01").cast("date"), (F.col("cliente_id") % 1100).cast("int"))))

produtos = (spark.range(1, N_PRODUTOS + 1).withColumnRenamed("id", "produto_id")
    .withColumn("nome_produto", F.concat(F.lit("Produto "), F.col("produto_id")))
    .withColumn("categoria", F.element_at(F.array(*[F.lit(x) for x in ["Eletrônicos", "Casa", "Moda", "Esportes", "Livros"]]),
                                        (F.pmod(F.col("produto_id"), F.lit(5)) + 1).cast("int")))
    .withColumn("preco_lista", F.round(F.lit(10.0) + (F.col("produto_id") % 5000) * F.lit(0.17), 2).cast("decimal(12,2)")))

vendas = (spark.range(1, N_VENDAS + 1).withColumnRenamed("id", "venda_id")
    .withColumn("cliente_id", (F.floor(F.rand(SEED) * N_CLIENTES) + 1).cast("long"))
    .withColumn("produto_id", (F.floor(F.rand(SEED + 1) * N_PRODUTOS) + 1).cast("long"))
    .withColumn("data_venda", F.date_add(F.lit("2024-01-01").cast("date"),
                                          F.floor(F.rand(SEED + 2) * 730).cast("int")))
    .withColumn("quantidade", (F.floor(F.rand(SEED + 3) * 5) + 1).cast("int"))
    .withColumn("preco_unitario", F.round(F.lit(10.0) + F.rand(SEED + 4) * 990.0, 2).cast("decimal(12,2)"))
    .withColumn("desconto_pct", F.round(F.rand(SEED + 5) * 0.25, 4).cast("decimal(5,4)"))
    .withColumn("canal", F.when(F.rand(SEED + 6) < 0.65, "online").otherwise("loja")))

for nome, df in [("clientes", clientes), ("produtos", produtos), ("vendas", vendas)]:
    (df.repartition(8).write.mode("overwrite").parquet(f"{RAW}/{nome}"))
    print(nome, df.count())

## 3. Bronze — ingestão e rastreabilidade

Lemos o landing em Parquet e persistimos em Delta. A coluna `arquivo_origem` ajuda a rastrear a procedência. A carga inicial usa `overwrite` para tornar o laboratório reexecutável; para produção, veja a seção de incrementalidade.

In [ ]:
for nome in ("clientes", "produtos", "vendas"):
    df = (spark.read.parquet(f"{RAW}/{nome}")
          .withColumn("data_ingestao", F.current_timestamp())
          .withColumn("arquivo_origem", F.input_file_name()))
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .save(f"{BRONZE}/{nome}"))
    print(f"bronze.{nome}: {spark.read.format('delta').load(f'{BRONZE}/{nome}').count():,}")

## 4. Silver — padronização, deduplicação e qualidade

Regras: identificadores positivos, campos obrigatórios, valores válidos, desconto entre 0 e 1, chaves estrangeiras existentes. Registros inválidos são enviados à **quarentena**; os válidos são deduplicados por chave de negócio, priorizando a ingestão mais recente.

Em um pipeline real, mantenha também métricas por regra de rejeição e acordos de qualidade de dados.

In [ ]:
bc = spark.read.format("delta").load(f"{BRONZE}/clientes")
bp = spark.read.format("delta").load(f"{BRONZE}/produtos")
bv = spark.read.format("delta").load(f"{BRONZE}/vendas")

def dedup(df, key):
    w = Window.partitionBy(key).orderBy(F.col("data_ingestao").desc(), F.col("arquivo_origem").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

sc = dedup(bc.filter("cliente_id > 0 AND nome IS NOT NULL AND email IS NOT NULL"), "cliente_id")
sp = dedup(bp.filter("produto_id > 0 AND nome_produto IS NOT NULL AND preco_lista >= 0"), "produto_id")

validacao = (bv.withColumn("motivo_rejeicao",
    F.when(F.col("venda_id").isNull() | (F.col("venda_id") <= 0), "venda_id_invalido")
     .when(F.col("cliente_id").isNull() | (F.col("cliente_id") <= 0), "cliente_id_invalido")
     .when(F.col("produto_id").isNull() | (F.col("produto_id") <= 0), "produto_id_invalido")
     .when(F.col("data_venda").isNull(), "data_ausente")
     .when(F.col("quantidade").isNull() | (F.col("quantidade") <= 0), "quantidade_invalida")
     .when(F.col("preco_unitario").isNull() | (F.col("preco_unitario") < 0), "preco_invalido")
     .when(F.col("desconto_pct").isNull() | (F.col("desconto_pct") < 0) | (F.col("desconto_pct") > 1), "desconto_invalido")))

pre_validas = dedup(validacao.filter("motivo_rejeicao IS NULL").drop("motivo_rejeicao"), "venda_id")
com_clientes = pre_validas.join(sc.select("cliente_id"), "cliente_id", "left_semi")
sv = com_clientes.join(sp.select("produto_id"), "produto_id", "left_semi")

# Quarentena inclui inválidos por regra e referências inexistentes.
invalidas = validacao.filter("motivo_rejeicao IS NOT NULL")
sem_cliente = pre_validas.join(sc.select("cliente_id"), "cliente_id", "left_anti").withColumn("motivo_rejeicao", F.lit("cliente_inexistente"))
sem_produto = com_clientes.join(sp.select("produto_id"), "produto_id", "left_anti").withColumn("motivo_rejeicao", F.lit("produto_inexistente"))
quarentena = invalidas.unionByName(sem_cliente).unionByName(sem_produto)

for nome, df in [("clientes", sc), ("produtos", sp), ("vendas", sv), ("quarentena_vendas", quarentena)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SILVER}/{nome}")
    print(f"silver.{nome}: {df.count():,}")

## 5. Gold — modelo dimensional e indicadores

As dimensões neste laboratório representam o **estado atual** (SCD tipo 1). Para histórico de atributos, seria necessário implementar SCD tipo 2. A fato contém a receita líquida calculada com precisão decimal.

In [ ]:
dim_clientes = sc.select("cliente_id", "nome", "estado", "data_cadastro")
dim_produtos = sp.select("produto_id", "nome_produto", "categoria", "preco_lista")
fato_vendas = (sv.withColumn("valor_bruto", (F.col("quantidade") * F.col("preco_unitario")).cast("decimal(18,2)"))
    .withColumn("valor_liquido", (F.col("quantidade") * F.col("preco_unitario") * (1 - F.col("desconto_pct"))).cast("decimal(18,2)"))
    .select("venda_id", "cliente_id", "produto_id", "data_venda", "canal", "quantidade", "preco_unitario", "desconto_pct", "valor_bruto", "valor_liquido"))

for nome, df in [("dim_clientes", dim_clientes), ("dim_produtos", dim_produtos), ("fato_vendas", fato_vendas)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{GOLD}/{nome}")
    df.createOrReplaceTempView(nome)

vendas_enriquecidas = (fato_vendas.join(dim_produtos.select("produto_id", "categoria"), "produto_id")
                      .join(dim_clientes.select("cliente_id", "estado"), "cliente_id"))

resumo_mensal = (vendas_enriquecidas.withColumn("mes", F.date_format("data_venda", "yyyy-MM"))
    .groupBy("mes", "canal")
    .agg(F.count("*").alias("total_vendas"), F.sum("quantidade").alias("unidades"),
         F.sum("valor_liquido").alias("receita_liquida"),
         F.round(F.avg("valor_liquido"), 2).alias("ticket_medio")))
receita_categoria = (vendas_enriquecidas.groupBy("categoria")
    .agg(F.count("*").alias("total_vendas"), F.sum("valor_liquido").alias("receita_liquida")))
receita_estado = (vendas_enriquecidas.groupBy("estado")
    .agg(F.count("*").alias("total_vendas"), F.sum("valor_liquido").alias("receita_liquida")))

for nome, df in [("resumo_mensal", resumo_mensal), ("receita_categoria", receita_categoria), ("receita_estado", receita_estado)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{GOLD}/{nome}")
    df.createOrReplaceTempView(nome)

spark.sql("""SELECT COUNT(*) AS vendas, SUM(quantidade) AS unidades,
                  SUM(valor_liquido) AS receita_total,
                  ROUND(AVG(valor_liquido),2) AS ticket_medio,
                  COUNT(DISTINCT cliente_id) AS clientes_com_compra
           FROM fato_vendas""").show(truncate=False)

## 6. Consultas analíticas com Spark SQL

In [ ]:
spark.sql("""SELECT mes, SUM(total_vendas) AS vendas, SUM(receita_liquida) AS receita
FROM resumo_mensal GROUP BY mes ORDER BY mes""").show(30, truncate=False)
spark.sql("SELECT * FROM receita_categoria ORDER BY receita_liquida DESC").show(truncate=False)
spark.sql("SELECT * FROM receita_estado ORDER BY receita_liquida DESC").show(truncate=False)

## 7. Visualização de indicadores

In [ ]:
import matplotlib.pyplot as plt
# Apenas os 24 meses agregados são coletados para o driver.
pdf = (resumo_mensal.groupBy("mes").agg(F.sum("receita_liquida").alias("receita"))
       .orderBy("mes").toPandas())
plt.figure(figsize=(12, 4))
plt.plot(pdf["mes"], pdf["receita"].astype(float), marker="o")
plt.xticks(rotation=60)
plt.title("Receita líquida mensal")
plt.xlabel("Mês")
plt.ylabel("Receita (R$)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Testes de qualidade e reconciliação

In [ ]:
assert bc.count() == N_CLIENTES
assert bp.count() == N_PRODUTOS
assert bv.count() == N_VENDAS
assert sc.count() == N_CLIENTES
assert sp.count() == N_PRODUTOS
assert sv.count() == N_VENDAS
assert quarentena.count() == 0
assert fato_vendas.count() == N_VENDAS
assert fato_vendas.filter("valor_liquido < 0").count() == 0
assert fato_vendas.select("venda_id").distinct().count() == N_VENDAS
assert resumo_mensal.agg(F.sum("total_vendas")).first()[0] == N_VENDAS
assert (fato_vendas.agg(F.sum("valor_liquido")).first()[0] ==
        resumo_mensal.agg(F.sum("receita_liquida")).first()[0])
print("✓ Todas as verificações de qualidade e reconciliação passaram.")

## 9. Exemplo de ingestão incremental com MERGE (Silver)

A função abaixo é **demonstrativa** e não altera as tabelas anteriores até ser chamada. Para dados novos, carregue primeiro um lote na Bronze, valide-o com as mesmas regras e só então faça o `MERGE`. `MERGE` permite *upsert* por chave de negócio, mas não garante sozinho exatamente-uma-vez no processo completo. Em produção, registre `batch_id`, origem, watermark e auditoria.

In [ ]:
from delta.tables import DeltaTable

def merge_silver_vendas(lote_validado):
    """Recebe lote já validado, deduplicado e com integridade referencial verificada."""
    destino = DeltaTable.forPath(spark, f"{SILVER}/vendas")
    (destino.alias("t").merge(lote_validado.alias("s"), "t.venda_id = s.venda_id")
     .whenMatchedUpdateAll(condition="s.data_ingestao >= t.data_ingestao")
     .whenNotMatchedInsertAll()
     .execute())

# Exemplo idempotente, desativado por padrão:
# merge_silver_vendas(sv.limit(100))

## 10. Inspeção do histórico Delta e próximos passos

In [ ]:
DeltaTable.forPath(spark, f"{GOLD}/fato_vendas").history(5).select("version", "timestamp", "operation").show(truncate=False)
print("Lakehouse disponível em:", ROOT)
# Ao encerrar o laboratório: spark.stop()

### Evoluções recomendadas para produção

- **Orquestração:** Airflow, Databricks Workflows ou outro agendador com retries e observabilidade.
- **Streaming:** Structured Streaming + Auto Loader (quando disponível), checkpoint e tratamento de eventos tardios.
- **Catálogo e governança:** Unity Catalog, Hive Metastore ou catálogo compatível; políticas de acesso e mascaramento de PII.
- **Qualidade:** métricas por lote, contratos de schema, tratamento de duplicidades e reconciliação automatizada.
- **Performance:** ajustar partições, compactação e otimização de arquivos conforme o mecanismo Delta disponível; evitar particionamento excessivo.
- **Histórico:** CDC e SCD tipo 2 para clientes/produtos, se atributos precisarem de histórico.
- **Operação:** caminhos de storage compartilhado (S3/ADLS/GCS), segredos seguros, retenção e políticas de recuperação.

**Exercícios:** (1) injetar vendas com chaves inválidas e inspecionar quarentena; (2) criar top 10 produtos por receita; (3) calcular taxa de recompra; (4) comparar execução com 8 e 32 partições; (5) implementar lote incremental com auditoria.